# Pipeline hoàn chỉnh — **2 GPU DDP**

Notebook này chạy **cùng pipeline** với `pipeline_1gpu.ipynb`, chỉ khác `WORLD_SIZE = 2`.

| Bước | Run | Global Batch | LR |
|------|-----|--------------|-----|
| 1 | baseline | 32 (16×2) | 1e-4 |
| 2 | lr_scaled | 32 (16×2) | 2e-4 |
| 3 | no_lr_scale | 32 (16×2) | 1e-4 (= baseline) |

Sau khi chạy xong: lưu `drive_backup/` (Colab: tải về hoặc copy lên Drive; Kaggle: upload dataset) để notebook còn lại đối chiếu Speedup.


In [ ]:
# ============================================================
# CẤU HÌNH DUY NHẤT KHÁC BIỆT GIỮA 2 NOTEBOOK
# ============================================================
WORLD_SIZE = 2
# ============================================================

## Bước 0: Setup môi trường

### Module `ddp_utils` (nhúng trong notebook)

Cell dưới chứa toàn bộ code `ddp_utils`, **ghi ra `ddp_utils.py` rồi import** (bắt buộc cho `mp.spawn`/DDP trên Colab — hàm worker không thể nằm trong `__main__` của notebook). Chạy cell này trước cell cấu hình đường dẫn bên dưới.

In [ ]:
!pip install -q timm

import importlib
import sys
from pathlib import Path

# mp.spawn không pickle được hàm định nghĩa trong notebook (__main__).
# Ghi ra ddp_utils.py rồi import như module thật.
if Path("/content").exists():
    _work_dir = Path("/content")
elif Path("/kaggle/working").exists():
    _work_dir = Path("/kaggle/working")
else:
    _work_dir = Path(".")

_DDP_UTILS_SOURCE = r'''
"""
PyTorch DDP utilities for scaling experiments.
Designed for Kaggle (2x T4) with mp.spawn — no torchrun required.
"""

from __future__ import annotations

import json
import os
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
import timm


@dataclass
class TrainConfig:
    """Training hyperparameters for one experiment run."""

    exp_name: str = "baseline"
    world_size: int = 1
    local_batch_size: int = 16
    learning_rate: float = 1e-4
    epochs: int = 5
    num_workers: int = 2
    model_name: str = "vit_large_patch16_224"
    num_classes: int = 100
    image_size: int = 224
    save_dir: str = "/kaggle/working/results"
    seed: int = 42
    use_amp: bool = True
    warmup_epochs: int = 1

    @property
    def global_batch_size(self) -> int:
        return self.local_batch_size * self.world_size


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def setup_ddp(rank: int, world_size: int, port: str = "12355") -> None:
    os.environ.setdefault("MASTER_ADDR", "localhost")
    os.environ.setdefault("MASTER_PORT", port)
    torch.cuda.set_device(rank)
    dist.init_process_group(
        backend="nccl", rank=rank, world_size=world_size, device_id=rank
    )


def cleanup_ddp() -> None:
    if dist.is_initialized():
        dist.destroy_process_group()


def build_transforms(image_size: int, train: bool = True):
    if train:
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.5071, 0.4867, 0.4408),
                    std=(0.2675, 0.2565, 0.2761),
                ),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.5071, 0.4867, 0.4408),
                std=(0.2675, 0.2565, 0.2761),
            ),
        ]
    )


def build_dataloaders(
    config: TrainConfig,
    rank: int,
    world_size: int,
    data_root: str = "/kaggle/working/data",
):
    train_ds = datasets.CIFAR100(
        root=data_root,
        train=True,
        download=True,
        transform=build_transforms(config.image_size, train=True),
    )
    val_ds = datasets.CIFAR100(
        root=data_root,
        train=False,
        download=True,
        transform=build_transforms(config.image_size, train=False),
    )

    train_sampler = (
        DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
        if world_size > 1
        else None
    )
    val_sampler = (
        DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)
        if world_size > 1
        else None
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config.local_batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=config.num_workers,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.local_batch_size,
        shuffle=False,
        sampler=val_sampler,
        num_workers=config.num_workers,
        pin_memory=True,
    )
    return train_loader, val_loader, train_sampler


def build_model(config: TrainConfig, device: torch.device) -> nn.Module:
    model = timm.create_model(
        config.model_name,
        pretrained=False,
        num_classes=config.num_classes,
    )
    return model.to(device)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[float, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast("cuda", enabled=True):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += images.size(0)

    if total == 0:
        return 0.0, 0.0
    return total_loss / total, correct / total


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    epoch: int,
    use_amp: bool,
) -> float:
    model.train()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / max(total, 1)


def _train_worker(rank: int, world_size: int, config: TrainConfig, shared: dict) -> None:
    set_seed(config.seed + rank)
    is_ddp = world_size > 1

    if is_ddp:
        setup_ddp(rank, world_size)
    device = torch.device(f"cuda:{rank}")

    train_loader, val_loader, train_sampler = build_dataloaders(
        config, rank, world_size
    )
    model = build_model(config, device)

    if is_ddp:
        model = DDP(model, device_ids=[rank], output_device=rank)

    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.05)
    scaler = GradScaler("cuda", enabled=config.use_amp)

    history: dict[str, list[float]] = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    # Synchronize before timing
    if is_ddp:
        dist.barrier()
    start = time.perf_counter()

    for epoch in range(config.epochs):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler, device, epoch, config.use_amp
        )
        val_loss, val_acc = evaluate(model, val_loader, device)
        epoch_time = time.perf_counter() - epoch_start

        if rank == 0:
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["epoch_time_sec"].append(epoch_time)
            print(
                f"[{config.exp_name}] epoch {epoch + 1}/{config.epochs} "
                f"loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
                f"time={epoch_time:.1f}s"
            )

        if is_ddp:
            dist.barrier()

    total_time = time.perf_counter() - start

    if rank == 0:
        save_path = Path(config.save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

        state_dict = model.module.state_dict() if is_ddp else model.state_dict()
        ckpt_path = save_path / f"{config.exp_name}_model.pt"
        torch.save(
            {
                "model_state_dict": state_dict,
                "config": asdict(config),
                "history": history,
            },
            ckpt_path,
        )

        result = {
            "exp_name": config.exp_name,
            "world_size": world_size,
            "local_batch_size": config.local_batch_size,
            "global_batch_size": config.global_batch_size,
            "learning_rate": config.learning_rate,
            "epochs": config.epochs,
            "total_time_sec": total_time,
            "avg_epoch_time_sec": sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"]),
            "final_val_acc": history["val_acc"][-1] if history["val_acc"] else 0.0,
            "best_val_acc": max(history["val_acc"]) if history["val_acc"] else 0.0,
            "history": history,
            "checkpoint": str(ckpt_path),
        }
        shared["result"] = result

        metrics_path = save_path / f"{config.exp_name}_metrics.json"
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump({k: v for k, v in result.items() if k != "history"}, f, indent=2)

        history_path = save_path / f"{config.exp_name}_history.json"
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)

    if is_ddp:
        cleanup_ddp()


def run_training(config: TrainConfig) -> dict[str, Any]:
    """Launch single-GPU or DDP training via mp.spawn (Kaggle-safe)."""
    world_size = config.world_size
    if world_size < 1:
        raise ValueError("world_size must be >= 1")
    if world_size > torch.cuda.device_count():
        raise RuntimeError(
            f"Requested {world_size} GPUs but only {torch.cuda.device_count()} available."
        )

    mp.set_start_method("spawn", force=True)
    manager = mp.Manager()
    shared: dict = manager.dict()

    if world_size == 1:
        _train_worker(0, 1, config, shared)
    else:
        mp.spawn(
            _train_worker,
            args=(world_size, config, shared),
            nprocs=world_size,
            join=True,
        )

    return dict(shared.get("result", {}))


def estimate_amdahl_parallel_fraction(speedup: float, num_gpus: int) -> float:
    """
    Given measured speedup S with N GPUs, solve for parallel fraction P:
        S = 1 / ((1 - P) + P/N)  =>  P = (S - 1) / (S - S/N)
    """
    if num_gpus <= 1 or speedup <= 1.0:
        return 0.0
    denom = speedup - (speedup / num_gpus)
    if abs(denom) < 1e-9:
        return 1.0
    p = (speedup - 1.0) / denom
    return max(0.0, min(1.0, p))


def theoretical_amdahl_speedup(parallel_fraction: float, num_gpus: int) -> float:
    serial = 1.0 - parallel_fraction
    return 1.0 / (serial + parallel_fraction / num_gpus)


def load_shared_result(exp_name: str, search_dirs: list[str | Path]) -> dict[str, Any] | None:
    """Load metrics + history exported by another notebook (via Kaggle Dataset / Drive)."""
    for base in search_dirs:
        base = Path(base)
        metrics_path = base / f"{exp_name}_metrics.json"
        history_path = base / f"{exp_name}_history.json"
        if not metrics_path.exists():
            continue
        with open(metrics_path, encoding="utf-8") as f:
            result = json.load(f)
        if history_path.exists():
            with open(history_path, encoding="utf-8") as f:
                result["history"] = json.load(f)
        return result
    return None


def backup_result(result: dict[str, Any], exp_name: str, results_dir: Path, drive_path: Path) -> None:
    """Copy metrics, history, and checkpoint to drive_backup for team sharing."""
    import shutil

    for suffix in ("_metrics.json", "_history.json", "_model.pt"):
        src = results_dir / f"{exp_name}{suffix}"
        if src.exists():
            shutil.copy(src, drive_path / src.name)


def get_pipeline_configs(
    world_size: int,
    *,
    base_lr: float = 1e-4,
    local_batch: int = 16,
    epochs: int = 5,
    save_dir: str = "/kaggle/working/results",
) -> list[TrainConfig]:
    """
    Cùng một pipeline 3 bước cho cả 1 GPU và 2 GPU DDP.

    Ý nghĩa tương đương giữa hai notebook:
      - baseline      : cấu hình chuẩn của môi trường (global batch = local × world_size)
      - lr_scaled     : global batch gấp đôi baseline-1GPU + LR tuyến tính (×2)
      - no_lr_scale   : cùng global batch với lr_scaled nhưng giữ LR baseline
    """
    tag = f"{world_size}gpu"
    common = dict(epochs=epochs, save_dir=save_dir)

    if world_size == 1:
        return [
            TrainConfig(
                exp_name=f"pipeline_baseline_{tag}",
                world_size=1,
                local_batch_size=local_batch,
                learning_rate=base_lr,
                **common,
            ),
            TrainConfig(
                exp_name=f"pipeline_lr_scaled_{tag}",
                world_size=1,
                local_batch_size=local_batch * 2,
                learning_rate=base_lr * 2,
                **common,
            ),
            TrainConfig(
                exp_name=f"pipeline_no_lr_scale_{tag}",
                world_size=1,
                local_batch_size=local_batch * 2,
                learning_rate=base_lr,
                **common,
            ),
        ]

    return [
        TrainConfig(
            exp_name=f"pipeline_baseline_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr,
            **common,
        ),
        TrainConfig(
            exp_name=f"pipeline_lr_scaled_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr * 2,
            **common,
        ),
        TrainConfig(
            exp_name=f"pipeline_no_lr_scale_{tag}",
            world_size=2,
            local_batch_size=local_batch,
            learning_rate=base_lr,
            **common,
        ),
    ]

'''

_utils_path = _work_dir / "ddp_utils.py"
_utils_path.write_text(_DDP_UTILS_SOURCE.strip() + "\n", encoding="utf-8")

sys.path.insert(0, str(_work_dir))
if "ddp_utils" in sys.modules:
    importlib.reload(sys.modules["ddp_utils"])
import ddp_utils
from ddp_utils import (
    backup_result,
    estimate_amdahl_parallel_fraction,
    get_pipeline_configs,
    load_shared_result,
    run_training,
    theoretical_amdahl_speedup,
)

print(f"Loaded ddp_utils from {_utils_path}")

In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

# Tự nhận Colab (/content) hoặc Kaggle (/kaggle/working)
if Path("/content").exists():
    WORK_DIR = Path("/content")
    OTHER_PIPELINE_INPUT = WORK_DIR / "shared" / "ddp-pipeline-1gpu"
elif Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
    OTHER_PIPELINE_INPUT = Path("/kaggle/input/ddp-pipeline-1gpu")
else:
    WORK_DIR = Path(".")
    OTHER_PIPELINE_INPUT = WORK_DIR / "shared" / "ddp-pipeline-1gpu"

RESULTS_DIR = WORK_DIR / "results"
DATA_DIR = WORK_DIR / "data"
DRIVE_PATH = WORK_DIR / "drive_backup"

for d in (RESULTS_DIR, DATA_DIR, DRIVE_PATH):
    d.mkdir(parents=True, exist_ok=True)

# run_training, get_pipeline_configs, ... đã import từ ddp_utils ở cell trên.

BASE_LR = 1e-4
LOCAL_BATCH = 16
EPOCHS = 5
NUM_GPUS = torch.cuda.device_count()

print(f"WORK_DIR = {WORK_DIR}")
print(f"WORLD_SIZE = {WORLD_SIZE}")
print(f"GPU khả dụng: {NUM_GPUS}")
!nvidia-smi -L
assert NUM_GPUS >= WORLD_SIZE, f"Cần >= {WORLD_SIZE} GPU, hiện có {NUM_GPUS}" 

## Bước 1–3: Huấn luyện (3 runs pipeline)

In [ ]:
configs = get_pipeline_configs(
    WORLD_SIZE,
    base_lr=BASE_LR,
    local_batch=LOCAL_BATCH,
    epochs=EPOCHS,
    save_dir=str(RESULTS_DIR),
)
RUN_KEYS = ["baseline", "lr_scaled", "no_lr_scale"]
results = {}

for key, cfg in zip(RUN_KEYS, configs):
    if key == "no_lr_scale" and WORLD_SIZE > 1:
        results[key] = results["baseline"]
        print(f"[{key}] Cùng cấu hình với baseline trên {WORLD_SIZE} GPU — tái sử dụng kết quả.")
        continue

    print("=" * 60)
    print(f"RUN: {key} | world_size={cfg.world_size} | local_bs={cfg.local_batch_size} | lr={cfg.learning_rate}")
    print("=" * 60)
    results[key] = run_training(cfg)
    backup_result(results[key], cfg.exp_name, RESULTS_DIR, DRIVE_PATH)
    print(f"  Thời gian: {results[key]['total_time_sec']:.1f}s | Best acc: {results[key]['best_val_acc']:.4f}")

## Bước 4: Bảng tổng hợp thời gian & accuracy

In [ ]:
summary_rows = []
for key in RUN_KEYS:
    r = results[key]
    summary_rows.append({
        "Run": key,
        "World Size": r["world_size"],
        "Local Batch": r["local_batch_size"],
        "Global Batch": r["global_batch_size"],
        "LR": r["learning_rate"],
        "Total Time (s)": round(r["total_time_sec"], 2),
        "Best Val Acc": round(r["best_val_acc"], 4),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(RESULTS_DIR / f"pipeline_summary_{WORLD_SIZE}gpu.csv", index=False)
shutil.copy(RESULTS_DIR / f"pipeline_summary_{WORLD_SIZE}gpu.csv", DRIVE_PATH / f"pipeline_summary_{WORLD_SIZE}gpu.csv")

## Bước 5: Biểu đồ Linear LR Scaling

In [ ]:
colors = {"baseline": "#4C72B0", "lr_scaled": "#55A868", "no_lr_scale": "#C44E52"}
labels = {
    "baseline": f"baseline (gb={results['baseline']['global_batch_size']}, lr={BASE_LR})",
    "lr_scaled": f"lr_scaled (gb={results['lr_scaled']['global_batch_size']}, lr={BASE_LR*2})",
    "no_lr_scale": f"no_lr_scale (gb={results['no_lr_scale']['global_batch_size']}, lr={BASE_LR})",
}

fig, ax = plt.subplots(figsize=(10, 5))
for key in RUN_KEYS:
    hist = results[key]["history"]["val_acc"]
    ax.plot(range(1, len(hist) + 1), hist, "o-", color=colors[key], label=labels[key])

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy")
ax.set_title(f"Linear LR Scaling — {WORLD_SIZE} GPU DDP")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = RESULTS_DIR / f"lr_scaling_{WORLD_SIZE}gpu.png"
plt.savefig(fig_path, dpi=150)
shutil.copy(fig_path, DRIVE_PATH / fig_path.name)
plt.show()

diff_scaled = results["lr_scaled"]["best_val_acc"] - results["baseline"]["best_val_acc"]
diff_no_scale = results["no_lr_scale"]["best_val_acc"] - results["baseline"]["best_val_acc"]
print(f"Chênh lệch Best Val Acc vs baseline:")
print(f"  lr_scaled:    {diff_scaled:+.4f}")
print(f"  no_lr_scale:  {diff_no_scale:+.4f}")

## Bước 6: Speedup & Định luật Amdahl (đối chiếu notebook còn lại)

So sánh thời gian **baseline** giữa 1 GPU và 2 GPU (cùng local batch/GPU).

In [ ]:
tag = f"{WORLD_SIZE}gpu"
my_baseline = results["baseline"]

if WORLD_SIZE == 1:
    other_tag, other_label = "2gpu", "2 GPU DDP"
    other_exp = "pipeline_baseline_2gpu"
else:
    other_tag, other_label = "1gpu", "1 GPU"
    other_exp = "pipeline_baseline_1gpu"

other_baseline = load_shared_result(other_exp, [OTHER_PIPELINE_INPUT, DRIVE_PATH, RESULTS_DIR])

if other_baseline:
    if WORLD_SIZE == 1:
        t1, t2 = my_baseline["total_time_sec"], other_baseline["total_time_sec"]
    else:
        t1, t2 = other_baseline["total_time_sec"], my_baseline["total_time_sec"]

    speedup = t1 / t2
    P_est = estimate_amdahl_parallel_fraction(speedup, num_gpus=2)

    print(f"Thời gian 1 GPU : {t1:.1f}s")
    print(f"Thời gian 2 GPU : {t2:.1f}s")
    print(f"Speedup: {speedup:.3f}x | P (Amdahl): {P_est:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(["1 GPU", "2 GPU"], [t1, t2], color=["#4C72B0", "#55A868"])
    axes[0].set_ylabel("Time (s)")
    axes[0].set_title("Baseline training time")

    gpu_range = np.arange(1, 5)
    axes[1].plot(gpu_range, [theoretical_amdahl_speedup(P_est, n) for n in gpu_range], "o-", label=f"Amdahl P={P_est:.2f}")
    axes[1].plot([1, 2], [1, speedup], "s--", color="red", label=f"Đo được {speedup:.2f}x")
    axes[1].plot(gpu_range, gpu_range, ":", color="gray", label="Linear")
    axes[1].set_xlabel("N GPU")
    axes[1].set_ylabel("Speedup")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()

    amdahl_path = RESULTS_DIR / "amdahl_analysis.png"
    plt.savefig(amdahl_path, dpi=150)
    shutil.copy(amdahl_path, DRIVE_PATH / amdahl_path.name)
    plt.show()

    report = {"speedup": speedup, "P": P_est, "t_1gpu": t1, "t_2gpu": t2}
    with open(RESULTS_DIR / "amdahl_report.json", "w") as f:
        json.dump(report, f, indent=2)
    shutil.copy(RESULTS_DIR / "amdahl_report.json", DRIVE_PATH / "amdahl_report.json")
else:
    print(f"Chưa có kết quả notebook {other_label}.")
    print(f"Upload drive_backup/ lên dataset ddp-pipeline-{other_tag} rồi Add Input vào notebook kia.")

## Bước 7: Kết luận (điền sau khi chạy)

- Thời gian baseline 2 GPU DDP: ___s
- LR scaling có cải thiện accuracy không?
- Speedup 1→2 GPU (nếu đã có cả 2 notebook): ___x